<a href="https://colab.research.google.com/github/Al-Amin95/PromptInjectionDetectionSystem/blob/model-train-comparison/notebooks/04_roberta_fine_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#RoBERTa Fine-Tuning
# Part 1 — Fine-Tuning Concept and Objective

RoBERTa fine-tuning means taking the pre-trained `roberta-base` model and training it further on the project dataset so it can classify prompts as either SAFE or INJECTION.

In this project, RoBERTa is the second transformer model being trained after DistilBERT. Its results will later be compared with DistilBERT and SecBERT.

## Task Type

Binary text classification.

## Labels

- 0 = SAFE
- 1 = INJECTION

## Notebook Objective

Fine-tune RoBERTa to detect prompt injection in user-supplied prompts.

#Part 2 - Notebook Setup

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

print("Drive ready")


# 2. Import basic modules
from pathlib import Path
import os
import sys
import time
import json
import shutil
import random
import platform
import subprocess


# 3. Configuration
repo_url = "https://github.com/Al-Amin95/PromptInjectionDetectionSystem.git"
branch = "model-train-comparison"
repo_dir = Path("/content/PromptInjectionDetectionSystem")

drive_base = Path("/content/drive/MyDrive/USW_Dissertation/Prompt-Injection-Detection-System-SHAP")

print("Repository URL:", repo_url)
print("Working branch:", branch)
print("Repository directory:", repo_dir)
print("Google Drive base:", drive_base)


# 4. Clone or update GitHub repository
if repo_dir.exists():
    print("Repository already exists. Updating repository...")
    os.chdir(repo_dir)

    subprocess.run(["git", "fetch", "origin"], check=True)
    subprocess.run(["git", "checkout", branch], check=True)
    subprocess.run(["git", "pull", "origin", branch], check=True)

else:
    print("Repository not found. Cloning repository...")
    os.chdir("/content")

    subprocess.run(["git", "clone", "-b", branch, repo_url], check=True)
    os.chdir(repo_dir)

print("Repository ready")
print("Current working directory:", Path.cwd())


# 5. Install required libraries
print("Installing required libraries...")

requirements_path = repo_dir / "requirements.txt"

if requirements_path.exists():
    print("requirements.txt found. Installing from requirements.txt")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements_path)],
        check=True
    )
else:
    print("requirements.txt not found. Installing main libraries only.")

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "transformers",
        "datasets",
        "accelerate",
        "evaluate",
        "scikit-learn",
        "pandas",
        "numpy",
        "matplotlib",
        "pyarrow",
        "safetensors"
    ],
    check=True
)

print("Libraries installed")


# 6. Import required packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import transformers
import datasets
import sklearn

print("Packages imported successfully")


# 7. Define repository paths
project_root = Path("/content/PromptInjectionDetectionSystem")

raw_data_dir = project_root / "data" / "raw"
processed_data_dir = project_root / "data" / "processed"
notebooks_dir = project_root / "notebooks"

repo_models_dir = project_root / "models"
repo_roberta_dir = repo_models_dir / "roberta"

results_dir = project_root / "results"
results_tables_dir = results_dir / "tables"
results_figures_dir = results_dir / "figures"
results_predictions_dir = results_dir / "predictions"
results_errors_dir = results_dir / "error_analysis"
results_logs_dir = results_dir / "logs"

results_tables_roberta_dir = results_tables_dir / "roberta"
results_figures_roberta_dir = results_figures_dir / "roberta"
results_predictions_roberta_dir = results_predictions_dir / "roberta"
results_errors_roberta_dir = results_errors_dir / "roberta"
results_logs_roberta_dir = results_logs_dir / "roberta"

train_path = processed_data_dir / "clean_train.parquet"
validation_path = processed_data_dir / "clean_validation.parquet"
test_path = processed_data_dir / "clean_test.parquet"

print("Repository paths defined")


# 8. Define Google Drive paths
drive_data_dir = drive_base / "data"
drive_models_dir = drive_base / "models"
drive_notebooks_dir = drive_base / "notebooks"
drive_results_dir = drive_base / "results"
drive_screenshots_dir = drive_base / "screenshots"
drive_shap_outputs_dir = drive_base / "shap_outputs"
drive_dissertation_evidence_dir = drive_base / "dissertation_evidence"
drive_github_backup_dir = drive_base / "github_repo_backup"

drive_distilbert_dir = drive_models_dir / "distilbert"
drive_roberta_dir = drive_models_dir / "roberta"
drive_secbert_dir = drive_models_dir / "secbert"

drive_results_tables_dir = drive_results_dir / "tables"
drive_results_figures_dir = drive_results_dir / "figures"
drive_results_predictions_dir = drive_results_dir / "predictions"
drive_results_errors_dir = drive_results_dir / "error_analysis"
drive_results_logs_dir = drive_results_dir / "logs"

drive_results_tables_roberta_dir = drive_results_tables_dir / "roberta"
drive_results_figures_roberta_dir = drive_results_figures_dir / "roberta"
drive_results_predictions_roberta_dir = drive_results_predictions_dir / "roberta"
drive_results_errors_roberta_dir = drive_results_errors_dir / "roberta"
drive_results_logs_roberta_dir = drive_results_logs_dir / "roberta"

print("Google Drive paths defined")


# 9. Create required folders
folders_to_create = [
    # Repository folders
    repo_models_dir,
    repo_roberta_dir,
    results_dir,
    results_tables_dir,
    results_figures_dir,
    results_predictions_dir,
    results_errors_dir,
    results_logs_dir,
    results_tables_roberta_dir,
    results_figures_roberta_dir,
    results_predictions_roberta_dir,
    results_errors_roberta_dir,
    results_logs_roberta_dir,

    # Google Drive folders
    drive_base,
    drive_data_dir,
    drive_models_dir,
    drive_notebooks_dir,
    drive_results_dir,
    drive_screenshots_dir,
    drive_shap_outputs_dir,
    drive_dissertation_evidence_dir,
    drive_github_backup_dir,
    drive_distilbert_dir,
    drive_roberta_dir,
    drive_secbert_dir,
    drive_results_tables_dir,
    drive_results_figures_dir,
    drive_results_predictions_dir,
    drive_results_errors_dir,
    drive_results_logs_dir,
    drive_results_tables_roberta_dir,
    drive_results_figures_roberta_dir,
    drive_results_predictions_roberta_dir,
    drive_results_errors_roberta_dir,
    drive_results_logs_roberta_dir,
]

for folder in folders_to_create:
    folder.mkdir(parents=True, exist_ok=True)

print("Required folders created")


# 10. Check prepared dataset files
print("\nDataset file check")
print("Train file exists:", train_path.exists(), "|", train_path)
print("Validation file exists:", validation_path.exists(), "|", validation_path)
print("Test file exists:", test_path.exists(), "|", test_path)


# 11. Check RoBERTa folders
print("\nRoBERTa folder check")
print("Repository RoBERTa folder exists:", repo_roberta_dir.exists(), "|", repo_roberta_dir)
print("Drive RoBERTa folder exists:", drive_roberta_dir.exists(), "|", drive_roberta_dir)


# 12. Check active Git branch
current_branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    text=True
).strip()

print("\nBranch check")
print("Expected branch:", branch)
print("Current branch:", current_branch)


# 13. Check Colab environment
try:
    import google.colab
    running_in_colab = True
except ImportError:
    running_in_colab = False

print("\nEnvironment check")
print("Running in Colab:", running_in_colab)
print("Python version:", sys.version.split()[0])
print("Platform:", platform.platform())
print("PyTorch version:", torch.__version__)
print("Transformers version:", transformers.__version__)
print("Datasets version:", datasets.__version__)
print("Scikit-learn version:", sklearn.__version__)


# 14. Final setup check
all_datasets_exist = train_path.exists() and validation_path.exists() and test_path.exists()
roberta_repo_folder_exists = repo_roberta_dir.exists()
roberta_drive_folder_exists = drive_roberta_dir.exists()
correct_branch = current_branch == branch

print("\nFinal setup check")
print("Correct branch:", correct_branch)
print("All prepared datasets exist:", all_datasets_exist)
print("RoBERTa repository folder exists:", roberta_repo_folder_exists)
print("RoBERTa Drive folder exists:", roberta_drive_folder_exists)
print("Running in Colab:", running_in_colab)

if correct_branch and all_datasets_exist and roberta_repo_folder_exists and roberta_drive_folder_exists and running_in_colab:
    print("\nPart 2 complete")
else:
    print("\nPart 2 needs checking")

#Part 3 — Define Paths and Output Folders


In [ ]:
from pathlib import Path
import os

project_root = Path("/content/PromptInjectionDetectionSystem")
drive_base = Path("/content/drive/MyDrive/USW_Dissertation/Prompt-Injection-Detection-System-SHAP")

processed_data_dir = project_root / "data" / "processed"

train_path = processed_data_dir / "clean_train.parquet"
validation_path = processed_data_dir / "clean_validation.parquet"
test_path = processed_data_dir / "clean_test.parquet"

roberta_model_dir = project_root / "models" / "roberta"
roberta_best_model_dir = roberta_model_dir / "best_model"
roberta_tokenizer_dir = roberta_model_dir / "tokenizer"

roberta_tables_dir = project_root / "results" / "tables" / "roberta"
roberta_figures_dir = project_root / "results" / "figures" / "roberta"
roberta_predictions_dir = project_root / "results" / "predictions" / "roberta"
roberta_error_analysis_dir = project_root / "results" / "error_analysis" / "roberta"
roberta_logs_dir = project_root / "results" / "logs" / "roberta"

drive_roberta_model_dir = drive_base / "models" / "roberta"
drive_roberta_best_model_dir = drive_roberta_model_dir / "best_model"
drive_roberta_tokenizer_dir = drive_roberta_model_dir / "tokenizer"

drive_roberta_tables_dir = drive_base / "results" / "tables" / "roberta"
drive_roberta_figures_dir = drive_base / "results" / "figures" / "roberta"
drive_roberta_predictions_dir = drive_base / "results" / "predictions" / "roberta"
drive_roberta_error_analysis_dir = drive_base / "results" / "error_analysis" / "roberta"
drive_roberta_logs_dir = drive_base / "results" / "logs" / "roberta"

temporary_training_output_dir = Path("/content/roberta_training_output")

folders_to_create = [
    roberta_model_dir,
    roberta_best_model_dir,
    roberta_tokenizer_dir,
    roberta_tables_dir,
    roberta_figures_dir,
    roberta_predictions_dir,
    roberta_error_analysis_dir,
    roberta_logs_dir,
    drive_roberta_model_dir,
    drive_roberta_best_model_dir,
    drive_roberta_tokenizer_dir,
    drive_roberta_tables_dir,
    drive_roberta_figures_dir,
    drive_roberta_predictions_dir,
    drive_roberta_error_analysis_dir,
    drive_roberta_logs_dir,
    temporary_training_output_dir,
]

for folder in folders_to_create:
    folder.mkdir(parents=True, exist_ok=True)

path_check = {
    "train_path": train_path.exists(),
    "validation_path": validation_path.exists(),
    "test_path": test_path.exists(),
    "roberta_best_model_dir": roberta_best_model_dir.exists(),
    "roberta_tokenizer_dir": roberta_tokenizer_dir.exists(),
    "roberta_tables_dir": roberta_tables_dir.exists(),
    "roberta_figures_dir": roberta_figures_dir.exists(),
    "roberta_predictions_dir": roberta_predictions_dir.exists(),
    "roberta_error_analysis_dir": roberta_error_analysis_dir.exists(),
    "roberta_logs_dir": roberta_logs_dir.exists(),
    "drive_roberta_best_model_dir": drive_roberta_best_model_dir.exists(),
    "drive_roberta_tokenizer_dir": drive_roberta_tokenizer_dir.exists(),
    "drive_roberta_tables_dir": drive_roberta_tables_dir.exists(),
    "drive_roberta_figures_dir": drive_roberta_figures_dir.exists(),
    "drive_roberta_predictions_dir": drive_roberta_predictions_dir.exists(),
    "drive_roberta_error_analysis_dir": drive_roberta_error_analysis_dir.exists(),
    "drive_roberta_logs_dir": drive_roberta_logs_dir.exists(),
    "temporary_training_output_dir": temporary_training_output_dir.exists(),
}

print("RoBERTa path check")

for name, exists in path_check.items():
    print(name + ":", exists)

all_paths_ready = all(path_check.values())

print("all RoBERTa paths ready:", all_paths_ready)

if all_paths_ready:
    print("part 3 complete")
else:
    print("part 3 needs checking")

#Part 4 — Check GPU Availability

In [ ]:
import torch
import pandas as pd
from pathlib import Path
from datetime import datetime

# 1. Define log paths
project_root = Path("/content/PromptInjectionDetectionSystem")
drive_base = Path("/content/drive/MyDrive/USW_Dissertation/Prompt-Injection-Detection-System-SHAP")

roberta_logs_dir = project_root / "results" / "logs" / "roberta"
drive_roberta_logs_dir = drive_base / "results" / "logs" / "roberta"

roberta_logs_dir.mkdir(parents=True, exist_ok=True)
drive_roberta_logs_dir.mkdir(parents=True, exist_ok=True)

# 2. Check CUDA availability
cuda_available = torch.cuda.is_available()

if cuda_available:
    device = torch.device("cuda")
    gpu_name = torch.cuda.get_device_name(0)
    gpu_count = torch.cuda.device_count()
    gpu_memory_total_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024 ** 3), 2)
    gpu_memory_allocated_gb = round(torch.cuda.memory_allocated(0) / (1024 ** 3), 2)
    gpu_memory_reserved_gb = round(torch.cuda.memory_reserved(0) / (1024 ** 3), 2)
else:
    device = torch.device("cpu")
    gpu_name = "No GPU available"
    gpu_count = 0
    gpu_memory_total_gb = 0
    gpu_memory_allocated_gb = 0
    gpu_memory_reserved_gb = 0

# 3. Create device summary
device_summary = pd.DataFrame([
    {
        "check_time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "cuda_available": cuda_available,
        "selected_device": str(device),
        "gpu_count": gpu_count,
        "gpu_name": gpu_name,
        "gpu_memory_total_gb": gpu_memory_total_gb,
        "gpu_memory_allocated_gb": gpu_memory_allocated_gb,
        "gpu_memory_reserved_gb": gpu_memory_reserved_gb,
        "torch_version": torch.__version__,
    }
])

# 4. Save device summary
repo_device_summary_path = roberta_logs_dir / "roberta_device_summary.csv"
drive_device_summary_path = drive_roberta_logs_dir / "roberta_device_summary.csv"

device_summary.to_csv(repo_device_summary_path, index=False)
device_summary.to_csv(drive_device_summary_path, index=False)

# 5. Print checks
print("CUDA available:", cuda_available)
print("Selected device:", device)
print("GPU count:", gpu_count)
print("GPU name:", gpu_name)
print("GPU total memory GB:", gpu_memory_total_gb)
print("GPU allocated memory GB:", gpu_memory_allocated_gb)
print("GPU reserved memory GB:", gpu_memory_reserved_gb)
print("PyTorch version:", torch.__version__)

print("Repo device summary saved:", repo_device_summary_path.exists(), "|", repo_device_summary_path)
print("Drive device summary saved:", drive_device_summary_path.exists(), "|", drive_device_summary_path)

if cuda_available:
    print("part 4 complete - GPU available")
else:
    print("part 4 complete - CPU only")
    print("Before training, change Colab runtime to GPU: Runtime > Change runtime type > T4 GPU")